# $\arctan(1/\sqrt{7})/\sqrt{7}$: historical CMF bounds

Each section constructs a CMF, walks its trajectory, and prints $\delta_N$ at source depth 1000 for the reduced rational approximant. A finite-depth value does not prove the published measure bound.

In [ ]:
# @title Install dependencies (run once per environment)
%pip -q install ramanujantools==0.0.8 sympy==1.14.0 mpmath==1.3.0 matplotlib==3.11.2

## Setup

The CMFs are constructed directly by `ramanujantools.cmf.pfq.pFq`. The starting readouts, walks, and approximants are exact.

In [ ]:
import sys

import matplotlib.pyplot as plt
import sympy as sp
from flint import arb, ctx

from ramanujantools import Matrix
from ramanujantools.cmf.pfq import pFq

if hasattr(sys, "set_int_max_str_digits"):
    sys.set_int_max_str_digits(0)
with ctx.workdps(7000):
    TARGET = (arb(1)/arb(7).sqrt()).atan()/arb(7).sqrt()
DEPTHS = list(range(10, 1001, 10))
x0, x1, y0 = sp.symbols("x0 x1 y0")
results = {}


def delta(approximant):
    """Finite-depth exponent of the reduced rational approximant."""
    p, q = int(approximant.p), int(approximant.q)
    with ctx.workprec(2*q.bit_length() + 128):
        error = abs(TARGET - arb(p)/q)
        return float((-1 - error.log()/arb(q).log()).mid())

## 1. Alladi–Robinson family: $\arctan(1/\sqrt{7})/\sqrt{7}$

[Quadratic integral and measure formula](https://www.math.ru.nl/~wzudilin/gat/oALLADI4.txt).
$$E_n=\int_0^1\frac{t^n(1-t)^n}{(t^2+7)^{n+1}}\,dt.$$
**CMF:** rank 2, ${}_2F_1(-1/7)$ ? **Start:** $(1/2,1;3/2)$ ? **Direction:** $(1,1;2)$. One step advances the source index by two; the plotted samples use $N/2$ steps.
With $c=\arctan(1/\sqrt{7})/\sqrt{7}$, $E_n=C_n+B_nc$ and $p_n/q_n=-C_n/B_n$. The source labels $c$ differently; the integral fixes its value. The displayed measure formula also uses its denominator lemma.
The exact identity
$$E_n=\frac{(n!)^2}{(2n+1)!\,7^{n+1}}{}_2F_1\!\left(\frac{n+1}{2},\frac{n+2}{2};n+\frac32;-\frac1{7}\right)$$
follows by expanding the quadratic denominator and cancelling an equal upper and lower parameter. The common scalar cancels in the rational readout.


In [ ]:
cmf = pFq(2, 1, -sp.Rational(1, 7))
start = {x0: sp.Rational(1, 2), x1: 1, y0: sp.Rational(3, 2)}
trajectory = {x0: 1, x1: 1, y0: 2}
# One lattice step advances the source index by two; all sampled N are even.
walks = cmf.walk(trajectory, [N//2 for N in DEPTHS], start)
approximants = [-1*M[1, 0]/(8*(2*M[0, 0]-M[1, 0])) for M in walks]

deltas = [delta(r) for r in approximants]
results["Alladi–Robinson family"] = (DEPTHS, deltas)
print(f"delta at depth 1000: {deltas[-1]:.12f}")

## 2. Heimonen–Matala-aho–Väänänen (1994): $\beta=4/3$

[Jacobi Padé construction, §§3–4 and examples](https://www.cambridge.org/core/services/aop-cambridge-core/content/view/249C98E58A8F5967C24F7999398BDBCE/S0004972700013691a.pdf/an-application-of-jacobi-type-polynomials-to-irrationality-measures.pdf). The published bound is $\mu_0=4.0298\ldots$.
Let $F(z)={}_2F_1(1,\tfrac12;\tfrac32;z)$ and $m=\lfloor(4/3)n\rfloor$. The Padé remainder is proportional to
$$J_0(n-\tfrac12,m,n+1)=\int_0^1 \frac{t^{n-1/2}(1-t)^m}{(1+t/7)^{n+1}}\,dt.$$
**CMF:** rank 2, ${}_2F_1(-1/7)$ ? **Start:** $(2,3/2;7/2)$, source $(n,m)=(1,1)$ ? **Direction:** $(3,3;7)$. The code gives the coefficient rows in the $(F,\theta F)$ basis; the target readout is $-(CM)_0/[7(BM)_0]$.
Euler integration gives $J_0=B(U+1,V+1){}_2F_1(W,U+1;U+V+2;-1/a)$. The beta factor cancels in the readout.


In [ ]:
cmf = pFq(2, 1, -sp.Rational(1, 7))
start = {x0: 2, x1: sp.Rational(3, 2), y0: sp.Rational(7, 2)}
C = Matrix([[-147, sp.Rational(2891, 8)]])
B = Matrix([[154, -378]])
trajectory = {x0: 3, x1: 3, y0: 7}
periods = list(range(3, 334, 3))
depths = [1 + 3*k for k in periods]
walks = cmf.walk(trajectory, periods, start)
approximants = [-(C*M)[0]/(7*(B*M)[0]) for M in walks]
deltas = [delta(r) for r in approximants]
results["Heimonen–Matala-aho–Väänänen (1994)"] = (depths, deltas)
print(f"delta at depth 1000: {deltas[-1]:.12f}")

## All constructions

$\delta_N$ against the source depth $N$.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5), constrained_layout=True)
for name, (depths, values) in results.items():
    ax.plot(depths, values, label=name)
ax.set(xlabel="$N$", ylabel=r"$\delta_N$")
ax.legend(frameon=False)
plt.show()

## Historical overview

| Construction | Published upper bound $\mu_0$ | $\delta_{\rm eq}$ | $\delta_{1000}$ | CMF |
|:--|--:|--:|--:|:--|
| Alladi–Robinson family | 4.856970593829 | 0.259270838518 | 0.268289913078 | rank-2 ${}_2F_1$ walk |
| Heimonen–Matala-aho–Väänänen | $4.0298\ldots$ | $\approx0.3301$ | 0.344250768433 | rank-2 ${}_2F_1$ walk |

$\delta_{\rm eq}=1/(\mu_0-1)$; the finite-depth values are diagnostics. The arithmetic denominator claims belong to the cited sources.